# SENTINEL — Colab Flow Test (sirf flow samajhne ke liye)

Ye notebook **asal weapon training nahi** hai. Iska maqsad sirf ye hai ke poora flow ek chhoti test run se verify ho jaye:

1. GPU milta hai ya nahi
2. Google Drive mount hota hai ya nahi
3. YOLOv8s training chalti hai ya nahi (chhota built-in `coco8` dataset, sirf 8 images, 5 epochs)
4. Training ke baad `best.pt` weights banti hain ya nahi
5. Weights Drive mai save hoti hain + unka SHA-256 hash nikalta hai ya nahi

Har cell ko bari bari chalana hai: cell select karke **Shift + Enter** dabao, output ane ka wait karo, phir agla cell.


## Cell 1 — GPU check
Agar GPU nahi mila to Runtime > Change runtime type > GPU select karke is cell ko dobara chalao.

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## Cell 2 — Google Drive mount
Popup ayega: apna Google account select karke **Allow** karna hai. Is se notebook ko tumhari Drive ke `SENTINEL_FYP` folder tak access mile gi.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/SENTINEL_FYP/test_run', exist_ok=True)
print("Drive ready: /content/drive/MyDrive/SENTINEL_FYP/test_run")


## Cell 3 — Ultralytics install
1–2 minute lagen ge. `Restart runtime` ka button aye to ignore karna hai, restart NAHI karna.

In [ ]:
%pip install -q ultralytics
import ultralytics
print("ultralytics:", ultralytics.__version__)


## Cell 4 — Mini training (TEST run)
YOLOv8s ko built-in `coco8` dataset per sirf **5 epochs** train karenge. Ye dataset khud download ho jayega (bohat chhota hai).
Is run ke numbers ka koi matlab nahi — maqsad sirf `best.pt` banta dekhna hai. 2–4 minute lagen ge.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")  # wahi model jo SENTINEL weapon detection ke liye use hoga
results = model.train(
    data="coco8.yaml",   # built-in sample dataset (test only — asal dataset Gun/Knife wala hai)
    epochs=5,
    imgsz=640,
    batch=8,
    name="sentinel_flow_test",
    verbose=True,
)
print("Training done.")


## Cell 5 — Evaluation metrics
Training ke baad model ko validate karke mAP / Precision / Recall print karenge — asal training mai bhi bilkul yehi metrics report mai jayen ge.

In [ ]:
from ultralytics import YOLO

best = YOLO("runs/detect/sentinel_flow_test/weights/best.pt")
metrics = best.val(data="coco8.yaml", split="val", verbose=False)
print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
print(f"Precision    : {metrics.box.mp:.4f}")
print(f"Recall       : {metrics.box.mr:.4f}")


## Cell 6 — Ek image per detection demo
Sample image per boxes draw karke dikhayega — yehi cheez asal project mai webcam/video per hogi.

In [ ]:
from ultralytics import YOLO
import glob
from IPython.display import Image, display

best = YOLO("runs/detect/sentinel_flow_test/weights/best.pt")
hits = glob.glob("/content/**/coco8/images/val/*.jpg", recursive=True)
assert hits, "coco8 sample image nahi mili — Cell 4 dobara chalao"
sample = sorted(hits)[0]
res = best.predict(sample, conf=0.25, save=True, project="flow_test_predict", name="demo", verbose=False)
out = glob.glob("flow_test_predict/demo/*.jpg")[0]
print("Detections:", [best.names[int(b.cls)] for b in res[0].boxes])
display(Image(out, width=640))


## Cell 7 — Weights ko Drive mai save + SHA-256 hash (TRACKING wala step)
`best.pt` ko Drive ke `SENTINEL_FYP/test_run/` folder mai copy karenge aur uska hash print karenge.
Asal project mai Muse isi hash + metrics ko repo mai run-record ke tor per commit kare ga — yehi hamara tracking system hai.

In [ ]:
import shutil, hashlib, datetime, pathlib

src = pathlib.Path("runs/detect/sentinel_flow_test/weights/best.pt")
dst_dir = pathlib.Path("/content/drive/MyDrive/SENTINEL_FYP/test_run")
stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
dst = dst_dir / f"flowtest_yolov8s_{stamp}_best.pt"
shutil.copy(src, dst)
# results.csv bhi copy kar do (training curves ka data)
csv_src = pathlib.Path("runs/detect/sentinel_flow_test/results.csv")
if csv_src.exists():
    shutil.copy(csv_src, dst_dir / f"flowtest_{stamp}_results.csv")

h = hashlib.sha256(dst.read_bytes()).hexdigest()
print("=" * 60)
print("===== COPY FROM HERE — ye block Muse ko bhej dena =====")
print(f"FLOW TEST OK")
print(f"Weights file : {dst.name}")
print(f"Size (MB)    : {dst.stat().st_size/1e6:.2f}")
print(f"SHA-256      : {h}")
print(f"mAP@0.5      : {metrics.box.map50:.4f}  (test run — numbers be-maqsad hain)")
print("===== COPY TO HERE =====")
print("=" * 60)


## Ho gaya!
Aakhri cell ka `COPY FROM HERE` wala block copy karke Muse ko chat mai bhej do.
Us se confirm ho jayega ke poora flow — GPU, Drive, training, weights, tracking — end-to-end chal raha hai.

**Note:** Colab band karne se `/content` wala sara data mit jata hai, lekin Drive mai save ki hui weights mehfooz rehti hain. Yehi wajah hai ke har run ke aakhir mai weights Drive mai copy karna lazmi step hai.